# **Pearl-AQI-Engine**: Model Experimentation & Tracking
**Objective:** Establish an automated MLflow tracking environment to test multiple regression algorithms against our engineered Hopsworks feature set. The goal is to predict `european_aqi` with the lowest possible RMSE and highest R² score.

In [1]:
!pip install hopsworks mlflow scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of opentelemetry-proto to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of opentelemetry-proto to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.6/807.6 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 kB 8.2 MB/s eta 0:00:0

In [2]:
import hopsworks
import pandas as pd
from google.colab import userdata

hw_api_key = userdata.get('HOPSWORKS_API_KEY')
project = hopsworks.login(api_key_value=hw_api_key)
fs = project.get_feature_store()

print("Fetching dataset...")
weather_fg = fs.get_feature_group(name="pearl_weather_features_v1", version=1)
df = weather_fg.show(500)
df = df.sort_values('time').reset_index(drop=True)

print(f"Data ready! Loaded {len(df)} hourly records.")



To ensure compatibility please install the latest bug fix release matching the minor version of your backend (4.7) by running 'pip install hopsworks==4.7.*'



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/32889
Fetching dataset...
Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (1.60s) 
Data ready! Loaded 500 hourly records.


### 1. **Data Splitting**
Machine learning models cannot be trained on the exact same data they are tested on. We must split our dataset into a "Training Set" (to learn patterns) and a "Testing Set" (to prove the model actually works).

In [3]:
from sklearn.model_selection import train_test_split
X = df.drop(columns=['time', 'european_aqi'])
y = df['european_aqi']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

print(f"Training shapes: {X_train.shape}, Testing shapes: {X_test.shape}")

Training shapes: (400, 7), Testing shapes: (100, 7)


### 2. **Baseline Model: Ridge Regression**
We establish a baseline using a penalized linear model. If a complex neural network or tree ensemble cannot beat this simple line, the complex model is not worth the computational cost.

In [4]:
import mlflow
import mlflow.sklearn
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
mlflow.set_experiment("Pearl_AQI_Model_Experiments")

with mlflow.start_run(run_name="Baseline_Ridge"):
    alpha_param = 1.0
    model_ridge = Ridge(alpha=alpha_param)
    model_ridge.fit(X_train, y_train)

    predictions = model_ridge.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)
    mlflow.log_param("alpha", alpha_param)
    mlflow.log_metric("MAE", mae)
    mlflow.log_metric("RMSE", rmse)
    mlflow.log_metric("R2", r2)
    mlflow.sklearn.log_model(model_ridge, "model")

    print(f"Ridge Baseline Logged -> R²: {r2:.3f} | MAE: {mae:.2f} | RMSE: {rmse:.2f}")

2026/05/30 10:06:37 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/30 10:06:37 INFO mlflow.store.db.utils: Updating database tables
2026/05/30 10:06:43 INFO mlflow.tracking.fluent: Experiment with name 'Pearl_AQI_Model_Experiments' does not exist. Creating a new experiment.
2026/05/30 10:06:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/30 10:06:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Ridge Baseline Logged -> R²: 0.050 | MAE: 10.31 | RMSE: 12.45


### 3. **Complex Model: Random Forest Regressor**
Testing an ensemble of decision trees to see if non-linear mathematical relationships exist within our features (e.g., specific interactions between momentum and hour).

In [5]:
from sklearn.ensemble import RandomForestRegressor

with mlflow.start_run(run_name="Complex_RandomForest"):
    trees = 100
    depth = 10
    model_rf = RandomForestRegressor(n_estimators=trees, max_depth=depth, random_state=42)
    model_rf.fit(X_train, y_train)
    predictions = model_rf.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)
    mlflow.log_param("n_estimators", trees)
    mlflow.log_param("max_depth", depth)
    mlflow.log_metric("MAE", mae)
    mlflow.log_metric("RMSE", rmse)
    mlflow.log_metric("R2", r2)
    mlflow.sklearn.log_model(model_rf, "model")

    print(f"Random Forest Logged -> R²: {r2:.3f} | MAE: {mae:.2f} | RMSE: {rmse:.2f}")

2026/05/30 10:06:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/30 10:06:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Random Forest Logged -> R²: 0.409 | MAE: 7.95 | RMSE: 9.81


In [6]:
experiment = mlflow.get_experiment_by_name("Pearl_AQI_Model_Experiments")
runs_df = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

leaderboard = runs_df[['tags.mlflow.runName', 'metrics.R2', 'metrics.RMSE', 'metrics.MAE']]
leaderboard = leaderboard.sort_values(by='metrics.R2', ascending=False).reset_index(drop=True)

print("MODEL LEADERBOARD ")
display(leaderboard)

MODEL LEADERBOARD 


,tags.mlflow.runName,metrics.R2,metrics.RMSE,metrics.MAE
0,Complex_RandomForest,0.409481,9.812419,7.950526
1,Baseline_Ridge,0.049543,12.448741,10.311386


### 4. **Model Registry** (Hopsworks)

Exporting our champion model (Random Forest) from the local Colab memory and uploading it to the secure Hopsworks Model Registry. We will also attach our winning metrics so future engineers know exactly how accurate this version is.

In [ ]:
import joblib
import os
import hopsworks
from google.colab import userdata

hw_api_key = userdata.get('HOPSWORKS_API_KEY')
project = hopsworks.login(api_key_value=hw_api_key)

mr = project.get_model_registry()

model_dir = "aqi_model_dir"
if not os.path.exists(model_dir):
    os.mkdir(model_dir)

model_path = os.path.join(model_dir, "random_forest_aqi.pkl")
joblib.dump(model_rf, model_path)

print("Model frozen locally. Preparing cloud upload...")
aqi_model = mr.python.create_model(
    name="pearl_aqi_random_forest",
    metrics={"R2": r2, "RMSE": rmse, "MAE": mae},
    description="Predicts european_aqi based on temporal and momentum features."
)

print("Uploading to Hopsworks Model Registry...")
aqi_model.save(model_dir)
print(f"\nSuccess! Model Version {aqi_model.version} is now live in the cloud.")



To ensure compatibility please install the latest bug fix release matching the minor version of your backend (4.7) by running 'pip install hopsworks==4.7.*'



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/32889
Model frozen locally. Preparing cloud upload...
Uploading to Hopsworks Model Registry...


  0%|          | 0/6 [00:00<?, ?it/s]

Moving model files from 'aqi_model_dir' to the model registry... This is the default behavior. Set keep_original_files=True to copy files instead.


Uploading /content/aqi_model_dir/random_forest_aqi.pkl: 0.000%|          | 0/2053745 elapsed<00:00 remaining<?

Model created, explore it at https://eu-west.cloud.hopsworks.ai:443/p/32889/models/pearl_aqi_random_forest/1

Success! Model Version 1 is now live in the cloud.


## **Algorithm Verification: The Model Bake-Off**
To ensure i selected the most robust engine for our CI/CD pipeline, i'll evaluate multiple algorithmic approaches. let's test linear models against advanced tree-based ensembles to find the optimal balance of predictive accuracy and variance reduction.



> *Ridge Regression AND Random Forest Regressor are already tested above*






### Lasso Regression
Testing an alternative linear model with L1 regularization, which inherently performs feature selection by pushing non-critical feature weights to zero.

In [8]:
from sklearn.linear_model import Lasso

with mlflow.start_run(run_name="Linear_Lasso"):
    alpha_param = 0.1
    model_lasso = Lasso(alpha=alpha_param)
    model_lasso.fit(X_train, y_train)

    predictions = model_lasso.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)
    mlflow.log_param("alpha", alpha_param)
    mlflow.log_metric("MAE", mae)
    mlflow.log_metric("RMSE", rmse)
    mlflow.log_metric("R2", r2)
    mlflow.sklearn.log_model(model_lasso, "model")

    print(f"Lasso Logged -> R²: {r2:.3f} | MAE: {mae:.2f} | RMSE: {rmse:.2f}")

2026/05/30 10:14:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/30 10:14:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Lasso Logged -> R²: 0.066 | MAE: 10.28 | RMSE: 12.34


### Decision Tree Regressor
Moving to non-linear architectures. Decision Trees capture complex interactions but are highly prone to variance (overfitting). let's log the `max_depth` to track this complexity.

In [9]:
from sklearn.tree import DecisionTreeRegressor

with mlflow.start_run(run_name="Tree_DecisionTree"):
    max_depth_param = 10
    model_dt = DecisionTreeRegressor(max_depth=max_depth_param, random_state=42)
    model_dt.fit(X_train, y_train)

    predictions = model_dt.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)
    mlflow.log_param("max_depth", max_depth_param)
    mlflow.log_metric("MAE", mae)
    mlflow.log_metric("RMSE", rmse)
    mlflow.log_metric("R2", r2)
    mlflow.sklearn.log_model(model_dt, "model")

    print(f"Decision Tree Logged -> R²: {r2:.3f} | MAE: {mae:.2f} | RMSE: {rmse:.2f}")

2026/05/30 10:15:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/30 10:15:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Decision Tree Logged -> R²: 0.067 | MAE: 8.95 | RMSE: 12.34


### Gradient Boosting Regressor
A boosting ensemble that builds shallow trees sequentially, focusing on minimizing the residual errors of the previous trees.

In [10]:
from sklearn.ensemble import GradientBoostingRegressor

with mlflow.start_run(run_name="Ensemble_GradientBoosting"):
    n_estimators_param = 100
    max_depth_param = 5
    model_gb = GradientBoostingRegressor(n_estimators=n_estimators_param, max_depth=max_depth_param, random_state=42)
    model_gb.fit(X_train, y_train)

    predictions = model_gb.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    # MLflow Tracking
    mlflow.log_param("n_estimators", n_estimators_param)
    mlflow.log_param("max_depth", max_depth_param)
    mlflow.log_metric("MAE", mae)
    mlflow.log_metric("RMSE", rmse)
    mlflow.log_metric("R2", r2)
    mlflow.sklearn.log_model(model_gb, "model")

    print(f"Gradient Boosting Logged -> R²: {r2:.3f} | MAE: {mae:.2f} | RMSE: {rmse:.2f}")


2026/05/30 10:16:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/30 10:16:56 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Gradient Boosting Logged -> R²: 0.457 | MAE: 7.44 | RMSE: 9.41


### Model Verification & Final Comparison
Now that all experiments are successfully logged in MLflow, I am compiling the results into a centralized dataframe to identify the highest performing architecture for the production pipeline.

In [12]:
experiment = mlflow.get_experiment_by_name("Pearl_AQI_Model_Experiments")
runs_df = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

leaderboard = runs_df[['tags.mlflow.runName', 'metrics.R2', 'metrics.RMSE', 'metrics.MAE']]
leaderboard = leaderboard.sort_values(by='metrics.R2', ascending=False).reset_index(drop=True)

print("MODEL LEADERBOARD ")
display(leaderboard)

MODEL LEADERBOARD 


,tags.mlflow.runName,metrics.R2,metrics.RMSE,metrics.MAE
0,Ensemble_GradientBoosting,0.457442,9.405507,7.441414
1,Complex_RandomForest,0.409481,9.812419,7.950526
2,Tree_DecisionTree,0.066619,12.336407,8.948333
3,Linear_Lasso,0.065911,12.341085,10.281142
4,Linear_Lasso,0.065911,12.341085,10.281142
5,Baseline_Ridge,0.049543,12.448741,10.311386


### **Verdict:** Random Forest Selected for Production Deployment

Based on our MLflow tracking leaderboard, the **Gradient Boosting** ensemble technically achieved the highest raw performance (R²: ~0.457, MAE: ~7.44). However, I selected the **Random Forest** (R²: ~0.409, MAE: ~7.95) as the final production model for the automated CI/CD pipeline due to the following engineering trade-offs:

1. **Unsupervised CI/CD Stability:** Gradient Boosting builds trees sequentially and is highly sensitive to hyperparameter drift over time. Random Forest builds independent trees in parallel, making it significantly more robust and stable for a daily, unsupervised GitHub Actions cron job.
2. **Computational Efficiency:** Because Random Forest trains its estimators in parallel, it consumes less compute time on the free-tier GitHub Actions runners compared to sequential boosting.
3. **Non-Linear Validation:** Both tree ensembles completely destroyed the linear models (R² ~0.06), successfully proving that the relationship between meteorological features and the Air Quality Index is highly complex and non-linear.

The Random Forest provides the perfect intersection of predictive accuracy, SHAP explainability, and automated pipeline stability.